# Brain_Scape — 03 Reconstruction

Build 3D meshes from registered brain volumes, label atlas regions, apply damage overlays, and export to multiple formats.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Load Registered Volume

Use the output from the preprocessing step.

In [ ]:
import nibabel as nib

data_dir = Path('../data')
job_id = 'notebook-demo'

# Try registered output first, fall back to processed
registered_path = data_dir / 'registered' / job_id / 'registered.nii.gz'
stripped_path = data_dir / 'processed' / job_id / 'stripped.nii.gz'

if registered_path.exists():
    scan_path = str(registered_path)
    print(f'Using registered volume: {scan_path}')
elif stripped_path.exists():
    scan_path = str(stripped_path)
    print(f'Using stripped volume (not registered): {scan_path}')
    print('Note: For best results, run 02_preprocessing.ipynb first.')
else:
    # Use any available NIfTI file
    nii_files = sorted(data_dir.rglob('*.nii.gz'))
    if nii_files:
        scan_path = str(nii_files[0])
        print(f'Using available scan: {scan_path}')
    else:
        scan_path = None
        print('No scan data found. Run 01_data_exploration.ipynb and 02_preprocessing.ipynb first.')

if scan_path:
    img = nib.load(scan_path)
    data = img.get_fdata()
    print(f'Volume shape: {data.shape}')

## 2. Build 3D Mesh

Use marching cubes to extract an isosurface from the brain volume.

In [ ]:
if scan_path:
    from reconstruction.mesh_builder import MeshBuilder

    output_dir = data_dir / 'outputs' / job_id
    output_dir.mkdir(parents=True, exist_ok=True)

    builder = MeshBuilder()
    mesh_data = builder.build(scan_path, str(output_dir / 'brain_mesh'))

    print(f'Mesh built successfully!')
    print(f'  Vertices:  {mesh_data["num_vertices"]:,}')
    print(f'  Faces:     {mesh_data["num_faces"]:,}')
    print(f'  Decimated: {mesh_data.get("decimated", False)}')
else:
    print('No scan data available.')

## 3. Region Labeling

Map mesh vertices to atlas regions (AAL, Brodmann, Desikan-Killiany).

In [ ]:
if scan_path:
    from reconstruction.region_labeler import RegionLabeler

    labeler = RegionLabeler()

    # Load atlas labels
    atlas_dir = data_dir / 'atlases'
    if atlas_dir.exists():
        regions = labeler.label_regions(
            mesh_path=str(output_dir / 'brain_mesh.obj'),
            volume_path=scan_path,
            atlas_dir=str(atlas_dir),
        )
        print(f'Regions labeled: {len(regions)} atlas regions identified')
        for r in regions[:10]:
            print(f'  {r["name"]}: {r.get("vertex_count", 0)} vertices')
        if len(regions) > 10:
            print(f'  ... and {len(regions) - 10} more regions')
    else:
        print('Atlas directory not found. Run: bash scripts/download_atlases.sh')
        print('Region labeling will use fallback mode.')

## 4. Damage Overlay

Apply severity coloring to the mesh based on analysis results.

This requires analysis output from `04_analysis.ipynb`. If not available, we'll demonstrate with synthetic data.

In [ ]:
if scan_path:
    from reconstruction.damage_overlay import DamageOverlay

    overlay = DamageOverlay()

    # Check for real analysis results
    analysis_path = output_dir / 'analysis.json'
    if analysis_path.exists():
        import json
        with open(analysis_path) as f:
            analysis = json.load(f)
        print('Using analysis results from 04_analysis.ipynb')
    else:
        # Synthetic damage data for demonstration
        analysis = {
            'regions': [
                {'anatomical_name': 'Left Hippocampus', 'severity_level': 4, 'severity_label': 'RED'},
                {'anatomical_name': 'Right Frontal Lobe', 'severity_level': 3, 'severity_label': 'ORANGE'},
                {'anatomical_name': 'Left Temporal Lobe', 'severity_level': 2, 'severity_label': 'YELLOW'},
            ],
            'overall_confidence': 0.87,
        }
        print('Using synthetic damage data for demonstration')

    damage_map = overlay.apply(
        mesh_path=str(output_dir / 'brain_mesh.obj'),
        analysis=analysis,
        volume_path=scan_path,
    )

    print(f'\nDamage overlay applied!')
    print(f'  Total vertices: {damage_map.get("total_vertices", 0):,}')
    print(f'  Regions with damage: {len(analysis.get("regions", []))}')

    # Show severity distribution
    severity_counts = {}
    for region in analysis.get('regions', []):
        label = region.get('severity_label', 'UNKNOWN')
        severity_counts[label] = severity_counts.get(label, 0) + 1

    print('\nSeverity distribution:')
    for label, count in sorted(severity_counts.items()):
        print(f'  {label}: {count} region(s)')

## 5. GIF Export

Generate a 360-degree rotational animation of the 3D brain.

In [ ]:
if scan_path:
    from reconstruction.gif_exporter import GIFExporter

    gif_exporter = GIFExporter(
        n_frames=36,     # 10-degree steps
        resolution=256,
        fps=8,
    )

    gif_path = str(output_dir / 'brain_rotation.gif')
    result = gif_exporter.export(
        volume_path=scan_path,
        output_path=gif_path,
    )
    print(f'GIF exported: {result}')
    print('This file can be embedded in clinical reports.')

## 6. Mesh Export (GLB, OBJ, STL)

Export the mesh in multiple formats for web, surgical tools, and 3D printing.

In [ ]:
if scan_path:
    from reconstruction.mesh_exporter import MeshExporter

    exporter = MeshExporter()

    mesh_path = str(output_dir / 'brain_mesh.obj')

    # Export GLB (for Three.js web viewer)
    glb_path = exporter.export(
        mesh_path=mesh_path,
        output_path=str(output_dir / 'brain.glb'),
        format='glb',
    )
    print(f'GLB exported: {glb_path}')

    # Export OBJ (for surgical tools)
    obj_path = exporter.export(
        mesh_path=mesh_path,
        output_path=str(output_dir / 'brain_view.obj'),
        format='obj',
    )
    print(f'OBJ exported: {obj_path}')

    # Export STL (for 3D printing)
    stl_path = exporter.export(
        mesh_path=mesh_path,
        output_path=str(output_dir / 'brain_print.stl'),
        format='stl',
    )
    print(f'STL exported: {stl_path}')

    # Show file sizes
    for fmt, path in [('GLB', glb_path), ('OBJ', obj_path), ('STL', stl_path)]:
        if path and os.path.exists(path):
            size_kb = os.path.getsize(path) / 1024
            print(f'  {fmt}: {size_kb:.1f} KB')

## 7. Summary

All reconstruction outputs are in `data/outputs/{job_id}/`:
- `brain_mesh.obj` — Full resolution mesh
- `brain.glb` — Draco-compressed for web viewer
- `brain_view.obj` — OBJ format for surgical tools
- `brain_print.stl` — STL for 3D printing
- `brain_rotation.gif` — 360-degree animation
- `damage_map.json` — Per-vertex severity colors

Proceed to `04_analysis.ipynb` to run the analysis pipeline.